# Sahayak — Fine-tune Gemma 4 on **Google Colab**

Self-contained: install → clone (dataset ships in the repo) → GPU-adaptive train → optional upload.

### Pick your runtime (Runtime → Change runtime type)
- **L4 or A100** (Colab Pro): native bf16 → trains **E4B** (best accuracy), no float32-upcast OOM, and it's fast.
- **T4** (free tier): no native bf16 → Unsloth keeps Gemma 4's vision/audio ops in float32, which OOMs **E4B** at load. This notebook auto-falls back to **E2B**, which fits with headroom.

This cell logic mirrors `sahayak_finetune.py`; nothing here trains without response masking.

**Before running:** add your Hugging Face token (accepted the Gemma license) as a Colab secret named `HF_TOKEN` — the key icon in the left sidebar → *Add new secret* → toggle *Notebook access*. Never paste the token in a cell.

In [ ]:
# Cell 1 — install the fine-tune stack (Unsloth pulls a Gemma-4-compatible transformers/trl/peft).
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], check=True)
pip('-U', 'unsloth', 'unsloth_zoo')
pip('-U', 'huggingface_hub[hf_transfer]')
print('\ninstall complete — if pip warns of a torch/transformers conflict: Runtime → Restart session, then rerun.')

In [ ]:
# Cell 2 — HF auth (from Colab secret HF_TOKEN) + clone the repo (dataset is inside it).
import os, subprocess
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets.')
except Exception as e:
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN (accepted the Gemma license): ')

REPO_URL = 'https://github.com/SivaNithishKumar/sankat-mochan.git'
repo_dir = '/content/sankat-mochan'
if os.path.isdir(os.path.join(repo_dir, '.git')):
    subprocess.run(['git', '-C', repo_dir, 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', repo_dir, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, repo_dir], check=True)
FT = os.path.join(repo_dir, 'finetune')
os.chdir(FT)
head = subprocess.run(['git', '-C', repo_dir, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('finetune dir:', FT, '\non commit   :', head)

In [ ]:
# Cell 3 — locate + validate the dataset (HARD GATE). The repo ships data/{train,val}.jsonl.
import glob, os, subprocess, sys
def find_one(names, roots=('data', '/content')):
    for r in roots:
        for n in names:
            hits = sorted(glob.glob(f'{r}/**/{n}', recursive=True))
            if hits: return hits[0]
    return None
TRAIN = find_one(['train.jsonl', 'all.jsonl'])
VAL   = find_one(['val.jsonl'])
HOLD  = find_one(['eval_holdout.jsonl', 'holdout.jsonl'])
print('train  :', TRAIN, '\nval    :', VAL, '\nholdout:', HOLD)
assert TRAIN and VAL, 'train.jsonl / val.jsonl not found (they ship in the repo under finetune/data/).'
for f in (TRAIN, VAL):
    assert subprocess.run([sys.executable, 'validate_dataset.py', f]).returncode == 0, f'validation FAILED: {f}'
print('\ndataset validation passed.')

In [ ]:
# Cell 4 — GPU-adaptive train. E4B on a native-bf16 GPU (L4/A100), else E2B (T4).
import torch, subprocess, sys
name = torch.cuda.get_device_name(0)
major = torch.cuda.get_device_capability(0)[0]
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
native_bf16 = major >= 8   # Ampere+ (A100 cc8.0, L4 cc8.9). T4 = 7.5 => False.
print(f'GPU: {name} | cc {major}.x | {vram:.1f} GB | native bf16 = {native_bf16}')

if native_bf16:
    MODEL, BS, GA = 'unsloth/gemma-4-E4B-it-unsloth-bnb-4bit', '2', '4'
    print('-> native bf16 GPU: training E4B (best accuracy).')
else:
    MODEL, BS, GA = 'unsloth/gemma-4-E2B-it-unsloth-bnb-4bit', '2', '4'
    print('-> no native bf16 (T4): Unsloth forces float32 on Gemma 4 => E4B OOMs at load. Using E2B.')

OUT = '/content/sahayak-out'
cmd = [sys.executable, 'sahayak_finetune.py',
       '--train', TRAIN, '--eval', VAL,
       '--model', MODEL, '--out', OUT,
       '--epochs', '3', '--lr', '2e-4',
       '--batch-size', BS, '--grad-accum', GA,
       '--lora-r', '32', '--lora-alpha', '32',
       '--max-seq-len', '1024',
       '--export-merged', '--export-gguf', 'q4_k_m']
print('\n' + ' '.join(cmd) + '\n')
subprocess.run(cmd, check=True)

In [ ]:
# Cell 5 (optional) — ship artifacts to a private HF repo (owner resolved from your token).
import os
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
user = api.whoami()['name']
repo = f'{user}/sahayak-gemma4'
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path='/content/sahayak-out', repo_id=repo)
print('uploaded ->', repo)
print('pull locally:  huggingface-cli download', repo, '--local-dir ./sahayak-gemma4')